# Import File and Packages

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
import pandas as pd

file_path = '/content/drive/MyDrive/550 Project/Regression Versions/simon/IBM Opensource Employee data about turnover.xlsx'
df = pd.read_excel(file_path)

print(f"Successfully loaded data from: {file_path}")
display(df.head())

Successfully loaded data from: /content/drive/MyDrive/550 Project/Regression Versions/simon/IBM Opensource Employee data about turnover.xlsx


,Age,Attrition,Attrition_1,Attrition_0,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,1,0,Travel_Rarely,1102,Sales,1,2,Life Sciences,...,1,80,0,8,0,1,6,4,0,5
1,49,No,0,1,Travel_Frequently,279,Research & Development,8,1,Life Sciences,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,1,0,Travel_Rarely,1373,Research & Development,2,2,Other,...,2,80,0,7,3,3,0,0,0,0
3,33,No,0,1,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,...,3,80,0,8,3,3,8,7,3,0
4,27,No,0,1,Travel_Rarely,591,Research & Development,2,1,Medical,...,4,80,1,6,3,3,2,2,2,2


## Package Installation

In [28]:
from __future__ import annotations

import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

## Config

In [30]:
SEED = 2026
TEST_SIZE = 0.25

import os

OUTPUT_DIR = "/content/drive/MyDrive/550 Project/Regression Versions/simon"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUT = {
    "segments_all": os.path.join(OUTPUT_DIR, "risk_profile_segments_tree.csv"),
    "segments_high": os.path.join(OUTPUT_DIR, "risk_profile_segments_tree_high_risk.csv"),
}

# Deck-ready selection thresholds
HIGH_RISK_MIN_LIFT = 1.5
HIGH_RISK_MIN_N_TEST = 10

# Script Overview

#Locked Risk Profiles (Constrained Decision Tree)

## Purpose
- Convert the **Step 2 driver story** into **interpretable, actionable risk segments** (i.e., “who is high risk?”).
- Produce **simple rules** that executives can understand (e.g., *OverTime = Yes AND JobLevel = 1 AND Age = Under 30*).
- Quantify each segment’s:
  - **Attrition rate** (how risky)
  - **Lift vs overall** (how much higher than baseline)
  - **Leavers captured** (how much of the problem this segment explains)

## Design choices (important)
- **Interpretability first**:
  - Constrained tree depth + minimum leaf size so segments are stable and readable.
- **One representation per concept**:
  - Avoid duplicates like raw+bucket together (prevents noisy splits and “double counting” the same story).
- **Aligned with Step 2 drivers**:
  - OverTime, Travel, Single, Promotion delay, Commute, Engagement/Satisfaction, Role growth, Mobility, Age/Tenure.

## What we run (high-level flow)
1) **Load data**  
   - Read the IBM attrition dataset
   - Create `AttritionFlag` (Yes=1 / No=0)

2) **Create locked buckets (consistent definitions)**
   - `AgeBucket` (Under 30 / 30–39 / 40–49 / 50+)
   - `PromotionDelayBucket` from `YearsSinceLastPromotion` (0–1 / 2–3 / 4–5 / 6+)
   - `RoleGrowthBucket` from `YearsInCurrentRole` (0–1 / 2–3 / 4–6 / 7+)
   - `EarlyTenureFlag` from `YearsAtCompany` (<3 years)

3) **Build the locked feature set (no duplicates)**
   - **Categorical features** (buckets + business attributes):
     - OverTime, BusinessTravel, MaritalStatus, JobLevel, Department, JobRole  
     - AgeBucket, PromotionDelayBucket, RoleGrowthBucket, EarlyTenureFlag
   - **Numeric features** (kept small for interpretability):
     - DistanceFromHome, JobInvolvement, JobSatisfaction, NumCompaniesWorked

4) **Train/Test split**
   - Stratified split so both sets have a similar attrition rate
   - This ensures reported segments are evaluated on true “holdout” data

5) **Fit a constrained decision tree (CV on TRAIN only)**
   - Use a pipeline:
     - OneHotEncode categorical features
     - Pass numeric features through
     - Train DecisionTreeClassifier (balanced class weights)
   - Tune only simple interpretability parameters (via GridSearchCV):
     - `max_depth` (limits complexity)
     - `min_samples_leaf` (prevents tiny/noisy segments)
   - Evaluate on HOLDOUT using:
     - ROC-AUC (ranking quality)
     - PR-AUC (better for imbalanced attrition)

6) **Extract segments and produce deck-ready outputs**
   - Convert each tree leaf into a **human-readable rule**
   - For each segment, compute on HOLDOUT:
     - `n_test` (segment size)
     - `attr_rate_test` (segment attrition rate)
     - `lift_test` (segment risk vs overall baseline)
     - `% leavers captured` (share of holdout leavers in that segment)
   - Print:
     - Top risk segments (ranked by attrition rate, with minimum size filter)
     - “Deck-ready” high-risk set (lift ≥ threshold and n_test ≥ threshold)

## Interpretation (how to read Step 3 results)
- **Attrition rate**: within this segment, what % left
- **Lift**: segment attrition rate ÷ overall attrition rate  
  - Lift = 2.0 means **2× higher risk than average**
- **% leavers captured**: of all leavers in the test set, what share fall into this segment  
  - Helps prioritize “high impact” segments vs just “high rate” small groups
- These segments are **pattern-based explanations**, not proof of causation.

## Notes
- Step 3 is **segmentation + storytelling**, not causal inference.
- We keep the tree constrained so the rules stay **stable** and **actionable**.
- We use **CV only to pick tree depth/leaf size**, not to over-optimize accuracy.

## Outputs (saved to your Drive folder)
- `risk_profile_segments_tree.csv`  
  - Full segment table (train + test metrics, rules, lift, leaver capture)
- `risk_profile_segments_tree_high_risk.csv`  
  - Filtered, deck-ready high-risk segments (lift + minimum size thresholds)

## Rerun instructions (Colab reminder)
- Set `DATA_PATH` to your Drive file path
- Set `OUTPUT_DIR` to your team folder (e.g., Simon folder)
- Run top-to-bottom; results print clean tables + write CSVs

# Helper

In [31]:
def _age_bucket(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    bins = [-np.inf, 29, 39, 49, np.inf]
    labels = ["Under 30", "30-39", "40-49", "50+"]
    return pd.cut(s, bins=bins, labels=labels)


def _promotion_delay_bucket(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    bins = [-np.inf, 1, 3, 5, np.inf]
    labels = ["0-1", "2-3", "4-5", "6+"]
    return pd.cut(s, bins=bins, labels=labels)


def _role_growth_bucket(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    bins = [-np.inf, 1, 3, 6, np.inf]
    labels = ["0-1", "2-3", "4-6", "7+"]
    return pd.cut(s, bins=bins, labels=labels)


def _early_tenure_flag(years_at_company: pd.Series) -> pd.Series:
    y = pd.to_numeric(years_at_company, errors="coerce")
    return (y < 3).astype("Int64")

# Function List

In [32]:

def prettify_rule(rule: str) -> str:
    """Convert OHE + numeric threshold rule text into readable segment rules."""

    if rule == "(ALL)":
        return rule

    def _ohe_to_text(feature: str, is_one: bool) -> str:
        # OneHotEncoder names look like "Col_Value".
        if "_" in feature:
            col, val = feature.split("_", 1)
        else:
            col, val = feature, ""

        # Nicer names
        rename = {
            "PromotionDelayBucket": "Promotion Delay",
            "RoleGrowthBucket": "Role Growth",
            "AgeBucket": "Age",
            "EarlyTenureFlag": "Early Tenure (<3y)",
        }
        col_disp = rename.get(col, col)

        if val == "":
            return f"{col_disp} {'=' if is_one else '≠'} 1"

        # Make EarlyTenureFlag more explicit
        if col == "EarlyTenureFlag":
            return f"{col_disp} = {'Yes' if is_one else 'No'}"

        # Make OverTime explicit if present (values are typically Yes/No)
        if col == "OverTime":
            return f"OverTime = {val}" if is_one else f"OverTime ≠ {val}"

        return f"{col_disp} = {val}" if is_one else f"{col_disp} ≠ {val}"

    out = []
    for part in rule.split(" AND "):
        part = part.strip()
        if "<=" in part:
            f, th = [x.strip() for x in part.split("<=")]
            thv = float(th)
            if abs(thv - 0.5) < 1e-9:
                out.append(_ohe_to_text(f, is_one=False))
            else:
                # Numeric threshold
                if f == "DistanceFromHome":
                    out.append(f"Commute (miles) ≤ {thv:.1f}")
                else:
                    out.append(f"{f} ≤ {thv:.3f}")
        elif ">" in part:
            f, th = [x.strip() for x in part.split(">")]
            thv = float(th)
            if abs(thv - 0.5) < 1e-9:
                out.append(_ohe_to_text(f, is_one=True))
            else:
                if f == "DistanceFromHome":
                    out.append(f"Commute (miles) > {thv:.1f}")
                else:
                    out.append(f"{f} > {thv:.3f}")
        else:
            out.append(part)

    return " AND ".join(out)


def get_leaf_rules(tree: DecisionTreeClassifier, feature_names: list[str]) -> dict[int, str]:
    """leaf_id -> rule string (AND conditions) in transformed feature space."""
    t = tree.tree_
    left = t.children_left
    right = t.children_right
    feat = t.feature
    thr = t.threshold

    parent: dict[int, int | None] = {0: None}
    cond: dict[int, str | None] = {0: None}

    stack = [0]
    while stack:
        node = stack.pop()
        if left[node] != -1:
            l = left[node]
            r = right[node]
            parent[l] = node
            parent[r] = node

            f = feature_names[feat[node]]
            th = thr[node]
            cond[l] = f"{f} <= {th:.6f}"
            cond[r] = f"{f} > {th:.6f}"

            stack.append(l)
            stack.append(r)

    leaves = np.where(left == -1)[0]
    rules: dict[int, str] = {}
    for leaf in leaves:
        parts: list[str] = []
        cur = leaf
        while parent[cur] is not None:
            parts.append(str(cond[cur]))
            cur = int(parent[cur])
        parts.reverse()
        rules[int(leaf)] = " AND ".join(parts) if parts else "(ALL)"
    return rules


@dataclass
class SegmentTable:
    leaf_id: int
    n: int
    attrition_rate: float
    leavers: int


def summarize_leaves(tree: DecisionTreeClassifier, X_trans, y: np.ndarray, rules: dict[int, str], baseline: float) -> pd.DataFrame:
    leaf_ids = tree.apply(X_trans)
    df_leaf = pd.DataFrame({"leaf_id": leaf_ids, "y": y})
    g = df_leaf.groupby("leaf_id").agg(n=("y", "size"), attrition_rate=("y", "mean"), leavers=("y", "sum")).reset_index()
    g["lift"] = g["attrition_rate"] / (baseline + 1e-12)
    g["rule"] = g["leaf_id"].map(rules)
    return g

# Data Load

In [33]:
try:
    df = pd.read_excel(file_path)
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"Could not find DATA_PATH: {file_path}. "
        "If using Colab+Drive, mount Drive and set DATA_PATH to the full Drive path."
    ) from e

# Basic column cleanup (safe for Colab)
df.columns = [str(c).strip() for c in df.columns]

# Target
if "Attrition" not in df.columns:
    raise ValueError("Expected column 'Attrition' in dataset.")

df["AttritionFlag"] = (df["Attrition"].astype(str).str.strip().str.lower() == "yes").astype(int)

# Locked buckets (consistent across Step 3)
df["AgeBucket"] = _age_bucket(df.get("Age"))
df["PromotionDelayBucket"] = _promotion_delay_bucket(df.get("YearsSinceLastPromotion"))
df["RoleGrowthBucket"] = _role_growth_bucket(df.get("YearsInCurrentRole"))
df["EarlyTenureFlag"] = _early_tenure_flag(df.get("YearsAtCompany"))

# Feature set

In [34]:
# - One rep per concept
# - Aligned with Step 2 story

CATEGORICAL_FEATURES = [
    "OverTime",
    "BusinessTravel",
    "MaritalStatus",
    "JobLevel",
    "Department",
    "JobRole",
    "AgeBucket",
    "PromotionDelayBucket",
    "RoleGrowthBucket",
    "EarlyTenureFlag",
]

NUMERIC_FEATURES = [
    "DistanceFromHome",
    "JobInvolvement",
    "JobSatisfaction",  # keep for now; can drop later if tree over-splits on survey vars
    "NumCompaniesWorked",
]

# Keep only columns that exist
cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]
num_cols = [c for c in NUMERIC_FEATURES if c in df.columns]

features = cat_cols + num_cols
model_df = df[features + ["AttritionFlag"]].dropna().copy()

X = model_df[features]
y = model_df["AttritionFlag"].astype(int)

baseline = float(y.mean())
print(f"Rows used: {len(model_df):,}")
print(f"Baseline attrition rate: {baseline:.2%}")

Rows used: 1,470
Baseline attrition rate: 16.12%


# Train/test Split

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)


# Fit Decision Tree

In [36]:
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

tree = DecisionTreeClassifier(random_state=SEED, class_weight="balanced")
pipe = Pipeline(steps=[("pre", pre), ("tree", tree)])

param_grid = {
    "tree__max_depth": [2, 3, 4, 5],
    "tree__min_samples_leaf": [30, 50, 75],
}

gs = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
)

gs.fit(X_train, y_train)
pipe = gs.best_estimator_
print("Best tree params:", gs.best_params_)

p_test = pipe.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, p_test)
pr = average_precision_score(y_test, p_test)

print("=== RISK PROFILE TREE (Constrained) ===")
print(f"Holdout ROC-AUC: {auc:.3f}")
print(f"Holdout PR-AUC:  {pr:.3f}")


Best tree params: {'tree__max_depth': 5, 'tree__min_samples_leaf': 30}
=== RISK PROFILE TREE (Constrained) ===
Holdout ROC-AUC: 0.746
Holdout PR-AUC:  0.375


# Extract Segments & Print Outputs

In [37]:
X_train_trans = pipe.named_steps["pre"].transform(X_train)
X_test_trans = pipe.named_steps["pre"].transform(X_test)

# Feature names
ohe = pipe.named_steps["pre"].named_transformers_["cat"]
cat_feature_names = list(ohe.get_feature_names_out(cat_cols)) if len(cat_cols) else []
feature_names = cat_feature_names + num_cols

fitted_tree: DecisionTreeClassifier = pipe.named_steps["tree"]

rules = get_leaf_rules(fitted_tree, feature_names)

train_sum = summarize_leaves(fitted_tree, X_train_trans, y_train.to_numpy(), rules, baseline)
train_sum = train_sum.rename(columns={
    "n": "n_train",
    "attrition_rate": "attr_rate_train",
    "leavers": "leavers_train",
    "lift": "lift_train",
})

test_sum = summarize_leaves(fitted_tree, X_test_trans, y_test.to_numpy(), rules, baseline)
test_sum = test_sum.rename(columns={
    "n": "n_test",
    "attrition_rate": "attr_rate_test",
    "leavers": "leavers_test",
    "lift": "lift_test",
})

total_leavers_test = int(y_test.sum())

merged = pd.merge(train_sum, test_sum, on=["leaf_id", "rule"], how="outer")
merged["pct_leavers_captured_test"] = merged["leavers_test"] / (total_leavers_test + 1e-12)
merged["pretty_rule"] = merged["rule"].fillna("(ALL)").apply(prettify_rule)

merged = merged.sort_values(["attr_rate_test", "n_test"], ascending=[False, False]).reset_index(drop=True)

# Print top segments (avoid tiny leaves)

def _print_table(df_show: pd.DataFrame, title: str, n: int = 12) -> None:
    print(f"\n=== {title} ===")
    if len(df_show) == 0:
        print("(none)")
        return
    cols = [
        "n_test",
        "attr_rate_test",
        "lift_test",
        "pct_leavers_captured_test",
        "n_train",
        "attr_rate_train",
        "lift_train",
        "pretty_rule",
    ]
    df_out = df_show.copy()
    # Format rates for readability
    for c in ["attr_rate_test", "attr_rate_train", "pct_leavers_captured_test"]:
        if c in df_out.columns:
            df_out[c] = df_out[c].astype(float)
    print(df_out[cols].head(n).to_string(index=False))


report_view = merged.dropna(subset=["n_test", "attr_rate_test"]).copy()
report_view = report_view[report_view["n_test"] >= 10]
_print_table(report_view, "TOP RISK SEGMENTS (ranked by TEST attrition rate; n_test ≥ 10)")

high_risk = report_view[(report_view["lift_test"] >= HIGH_RISK_MIN_LIFT) & (report_view["n_test"] >= HIGH_RISK_MIN_N_TEST)].copy()
_print_table(high_risk, f"HIGH-RISK SEGMENTS (deck-ready; lift_test ≥ {HIGH_RISK_MIN_LIFT} and n_test ≥ {HIGH_RISK_MIN_N_TEST})")

# Save CSVs
merged.to_csv(OUT["segments_all"], index=False)
high_risk.to_csv(OUT["segments_high"], index=False)

print(f"\nSaved segments table: {OUT['segments_all']}")
print(f"Saved high-risk segments table: {OUT['segments_high']}")

# Quick recap for teammates
print("\n=== STEP 3 RECAP (for teammates) ===")
print("- Built a constrained decision tree on bucketed, non-duplicative features aligned with Step 2 drivers.")
print("- Reported leaf segments on HOLDOUT with size, attrition rate, lift, and % of leavers captured.")
print("- Exported full segment table + a deck-ready high-risk subset.")


=== TOP RISK SEGMENTS (ranked by TEST attrition rate; n_test ≥ 10) ===
 n_test  attr_rate_test  lift_test  pct_leavers_captured_test  n_train  attr_rate_train  lift_train                                                                                                              pretty_rule
     14        0.571429   3.544304                   0.135593       53         0.698113    4.330069                                                                       OverTime = Yes AND JobLevel = 1 AND Age = Under 30
     15        0.466667   2.894515                   0.118644       30         0.433333    2.687764                                        OverTime = Yes AND JobLevel = 1 AND Age ≠ Under 30 AND NumCompaniesWorked > 1.500
     11        0.454545   2.819333                   0.084746       33         0.363636    2.255466                                        OverTime = Yes AND JobLevel = 1 AND Age ≠ Under 30 AND NumCompaniesWorked ≤ 1.500
     14        0.357143   2.215190          